# Markov Baseline
Simple first-order Markov Chain on observable states.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

df = pd.read_csv(Path('../data/event.csv'))
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
state_map = {'view': 'Browse', 'addtocart': 'Add_to_Cart', 'transaction': 'Purchase'}
df['state'] = df['event'].map(state_map)
df = df.sort_values(['visitorid', 'timestamp']).reset_index(drop=True)
df.head()

,visitorid,timestamp,event,state
0,101,2015-06-02 05:02:12.117,view,Browse
1,101,2015-06-02 05:03:12.117,view,Browse
2,101,2015-06-02 05:04:12.117,addtocart,Add_to_Cart
3,101,2015-06-02 05:05:12.117,transaction,Purchase
4,102,2015-06-02 05:18:52.117,view,Browse


In [2]:
sequences = df.groupby('visitorid')['state'].apply(list).apply(lambda s: s + ['Exit'])
pairs = []
for seq in sequences:
    pairs.extend(list(zip(seq[:-1], seq[1:])))

t = pd.DataFrame(pairs, columns=['from_state', 'to_state'])
counts = pd.crosstab(t['from_state'], t['to_state'])
probs = counts.div(counts.sum(axis=1), axis=0)

print('Transition counts:')
print(counts)
print('\nTransition probabilities:')
print(probs.round(3))

plt.figure(figsize=(6, 4))
sns.heatmap(probs, annot=True, fmt='.2f', cmap='Blues')
plt.title('Markov Transition Heatmap')
plt.tight_layout()
plt.show()

Transition counts:
to_state     Add_to_Cart  Browse  Exit  Purchase
from_state                                      
Add_to_Cart            0       0     4         3
Browse                 7       7     3         0
Purchase               0       0     3         0

Transition probabilities:
to_state     Add_to_Cart  Browse   Exit  Purchase
from_state                                       
Add_to_Cart        0.000   0.000  0.571     0.429
Browse             0.412   0.412  0.176     0.000
Purchase           0.000   0.000  1.000     0.000
